# Shoko Cloud Indexer for Google Drive

This notebook computes file hashes (`ED2K`, `CRC32`, `MD5`, `SHA1`) using the native `librhash` library **with real-time progress bars** (overall GB progress + per-file progress, MB/s speed, and ETA) and extracts `MediaInfo` for anime stored on Google Drive, saving the output to:
- `_shoko_cloud_index.json` — full hash and MediaInfo manifest;
- `_shoko_cloud_index.meta.json` — tiny metadata beacon (~160 bytes: `updated_at`, `file_count`, `index_size`, `index_sha1`) polled by the `ShokoCloudBridge` plugin so the main index is only downloaded when changes occur.

**How it works:**
1. **Incremental:** previously indexed files are skipped in milliseconds — the drive is never re-hashed from scratch.
2. **Move/Rename detection:** if you rename a folder or move episodes inside your Anime root, the script matches files by filename + exact byte size and reuses existing hashes without re-reading the file.
3. **Crash-safe:** both `_shoko_cloud_index.json` and `_shoko_cloud_index.meta.json` are saved atomically after every single file.
4. **Auto-sync in Shoko Server:** the `ShokoCloudBridge` plugin checks the 160-byte `_shoko_cloud_index.meta.json` and imports new entries into Shoko & Jellyfin without downloading video files.

In [ ]:
#@title Step 1. Mount Google Drive & Install Dependencies (librhash + mediainfo)
#@markdown Path to your Anime folder relative to `/content/drive/` (e.g. `MyDrive/Anime` or `Shareddrives/MySharedDrive/Anime`):
GDRIVE_ANIME_PATH = "MyDrive/Anime" #@param {type:"string"}

import os
import subprocess
from pathlib import Path
from google.colab import drive

print("[1/2] Installing native utilities rhash (librhash0) and mediainfo...")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "rhash", "librhash0", "mediainfo"], check=True)

print("[2/2] Mounting Google Drive...")
drive.mount("/content/drive")

ANIME_DIR = None
candidates = []
if GDRIVE_ANIME_PATH.strip():
    p = Path(GDRIVE_ANIME_PATH.strip())
    candidates.append(p if p.is_absolute() else (Path("/content/drive") / p))

shareddrives_root = Path("/content/drive/Shareddrives")
if shareddrives_root.exists():
    for sd in sorted(shareddrives_root.iterdir()):
        if (sd / "Anime").exists():
            candidates.append(sd / "Anime")

mydrive_anime = Path("/content/drive/MyDrive/Anime")
if mydrive_anime.exists():
    candidates.append(mydrive_anime)

for c in candidates:
    if c.exists():
        ANIME_DIR = c
        break

if ANIME_DIR:
    print(f"\n[+] Anime folder found: {ANIME_DIR}")
else:
    print("\n[!] Warning: specified folder was not found. Available drives in /content/drive:")
    if shareddrives_root.exists():
        for sd in shareddrives_root.iterdir():
            print("  - Shareddrives/" + sd.name)


In [ ]:
#@title Step 2. Run Incremental Hashing (with live progress bars & .meta.json)
#@markdown Optional subfolder filter (leave empty to scan the entire Anime folder):
SUBFOLDER_FILTER = "" #@param {type:"string"}
PRUNE_DELETED = True #@param {type:"boolean"}

import ctypes
import ctypes.util
import datetime
import hashlib
import json
import os
import shutil
import subprocess
import time
from pathlib import Path
from tqdm.auto import tqdm

VIDEO_EXTENSIONS = {
    ".mkv", ".avi", ".mp4", ".mov", ".ogm",
    ".wmv", ".mpg", ".mpeg", ".mk3d", ".m4v"
}
INDEX_FILENAME = "_shoko_cloud_index.json"
META_FILENAME = "_shoko_cloud_index.meta.json"

RHASH_CRC32 = 0x01
RHASH_MD5   = 0x04
RHASH_SHA1  = 0x08
RHASH_ED2K  = 0x80
RHPR_HEX = 0x02
RHPR_UPPERCASE = 0x08
ALL_IDS = RHASH_ED2K | RHASH_CRC32 | RHASH_MD5 | RHASH_SHA1

def load_librhash():
    for p in [
        "/usr/lib/x86_64-linux-gnu/librhash.so.0",
        "/usr/lib/x86_64-linux-gnu/librhash.so",
        ctypes.util.find_library("rhash"),
    ]:
        if not p:
            continue
        try:
            lib = ctypes.CDLL(p)
            lib.rhash_library_init()
            lib.rhash_init.argtypes = [ctypes.c_uint]
            lib.rhash_init.restype = ctypes.c_void_p
            lib.rhash_update.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_size_t]
            lib.rhash_final.argtypes = [ctypes.c_void_p, ctypes.c_void_p]
            lib.rhash_print.argtypes = [ctypes.c_char_p, ctypes.c_void_p, ctypes.c_uint, ctypes.c_int]
            lib.rhash_free.argtypes = [ctypes.c_void_p]
            return lib
        except Exception:
            continue
    raise RuntimeError("Failed to load librhash.so.0!")

LIBRHASH = load_librhash()

def compute_rhash_with_progress(filepath: str, filesize: int, desc: str, overall_bar=None):
    ctx = LIBRHASH.rhash_init(ALL_IDS)
    chunk_size = 8 * 1024 * 1024  # 8 MB read buffer
    buf = bytearray(chunk_size)
    c_buf = (ctypes.c_char * chunk_size).from_buffer(buf)
    try:
        with tqdm(
            total=filesize,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=desc,
            leave=False,
            dynamic_ncols=True,
        ) as pbar:
            with open(filepath, "rb") as f:
                while True:
                    n = f.readinto(buf)
                    if not n:
                        break
                    LIBRHASH.rhash_update(ctx, ctypes.byref(c_buf), n)
                    pbar.update(n)
                    if overall_bar is not None:
                        overall_bar.update(n)

        LIBRHASH.rhash_final(ctx, None)
        out = ctypes.create_string_buffer(130)
        res = []
        for hid in (RHASH_ED2K, RHASH_CRC32, RHASH_MD5, RHASH_SHA1):
            LIBRHASH.rhash_print(out, ctx, hid, RHPR_HEX | RHPR_UPPERCASE)
            res.append(out.value.decode("ascii").upper())
        return res[0], res[1], res[2], res[3]
    finally:
        LIBRHASH.rhash_free(ctx)

def compute_mediainfo(filepath: str):
    try:
        cmd = ["mediainfo", "--OUTPUT=JSON", filepath]
        res = subprocess.run(cmd, capture_output=True, text=True, check=True)
        out = res.stdout.strip()
        if out.startswith("{"):
            return json.loads(out)
    except Exception as e:
        print(f"  [!] Failed to extract MediaInfo: {e}")
    return None

def load_index(index_path: Path) -> dict:
    if not index_path.exists():
        return {"version": 1, "updated_at": None, "files": {}}
    with open(index_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if "files" not in data or not isinstance(data["files"], dict):
        return {"version": 1, "updated_at": None, "files": data}
    return data

def save_index_and_meta_atomic(index_path: Path, manifest: dict):
    now_iso = datetime.datetime.now(datetime.timezone.utc).isoformat()
    manifest["updated_at"] = now_iso
    payload_bytes = json.dumps(manifest, ensure_ascii=False, indent=2).encode("utf-8")
    sha1_hex = hashlib.sha1(payload_bytes).hexdigest().upper()

    tmp_path = index_path.with_suffix(".json.tmp")
    with open(tmp_path, "wb") as f:
        f.write(payload_bytes)
    os.replace(tmp_path, index_path)

    meta_path = index_path.parent / META_FILENAME
    meta_obj = {
        "version": 1,
        "updated_at": now_iso,
        "file_count": len(manifest.get("files", {})),
        "index_size": len(payload_bytes),
        "index_sha1": sha1_hex,
    }
    meta_tmp = meta_path.with_suffix(".json.tmp")
    with open(meta_tmp, "w", encoding="utf-8") as f:
        json.dump(meta_obj, f, ensure_ascii=False)
    os.replace(meta_tmp, meta_path)

if not ANIME_DIR or not Path(ANIME_DIR).exists():
    raise RuntimeError("Please run Step 1 first to mount Google Drive!")

anime_root = Path(ANIME_DIR).resolve()
index_path = anime_root / INDEX_FILENAME
meta_path = anime_root / META_FILENAME
manifest = load_index(index_path)
existing_files = manifest.setdefault("files", {})

scan_root = (anime_root / SUBFOLDER_FILTER.strip("/")) if SUBFOLDER_FILTER.strip() else anime_root
if not scan_root.exists():
    raise FileNotFoundError(f"Subfolder not found: {scan_root}")

print(f"[*] Walking file tree in: {scan_root}")
discovered = {}
for dirpath, dirnames, filenames in os.walk(scan_root):
    dirnames[:] = [
        d for d in sorted(dirnames)
        if not d.startswith(".") and d not in ("$RECYCLE.BIN", "-Archives")
    ]
    for fname in sorted(filenames):
        ext = os.path.splitext(fname)[1].lower()
        if ext not in VIDEO_EXTENSIONS:
            continue
        full_path = Path(dirpath) / fname
        rel_path = full_path.relative_to(anime_root).as_posix()
        st = full_path.stat()
        if st.st_size > 0:
            discovered[rel_path] = (full_path, st.st_size, st.st_mtime)

disappeared_by_name_size = {}
pruned_count = 0
if not SUBFOLDER_FILTER.strip():
    for old_rel, old_entry in list(existing_files.items()):
        if old_rel not in discovered:
            old_name = os.path.basename(old_rel)
            old_size = int(old_entry.get("size", 0))
            if old_size > 0 and old_entry.get("ed2k"):
                disappeared_by_name_size[(old_name, old_size)] = old_entry
            if PRUNE_DELETED:
                del existing_files[old_rel]
                pruned_count += 1

to_hash = []
skipped = 0
relocated = 0

for rel_path, (full_path, size, mtime) in discovered.items():
    cur = existing_files.get(rel_path)
    if cur and int(cur.get("size", -1)) == size and cur.get("ed2k"):
        skipped += 1
        continue

    fname = os.path.basename(rel_path)
    moved_entry = disappeared_by_name_size.get((fname, size))
    if moved_entry and moved_entry.get("ed2k"):
        existing_files[rel_path] = {
            **moved_entry,
            "size": size,
            "mtime": round(mtime, 2),
        }
        relocated += 1
        continue

    to_hash.append((rel_path, full_path, size, mtime))

total_new_bytes = sum(x[2] for x in to_hash)
total_new_gb = total_new_bytes / (1024 ** 3)
print(f"[*] Video files discovered: {len(discovered)}")
print(f"    - Already in index (skipped): {skipped}")
print(f"    - Relocated (hash reused without reading): {relocated}")
print(f"    - Removed from index: {pruned_count}")
print(f"    - New files to hash: {len(to_hash)} ({total_new_gb:.2f} GB)\n")

if relocated > 0 or pruned_count > 0 or (index_path.exists() and not meta_path.exists()):
    save_index_and_meta_atomic(index_path, manifest)

if not to_hash:
    print(f"[+] Everything is up to date! Index: {index_path} | Meta: {meta_path}")
else:
    overall_bar = tqdm(
        total=total_new_bytes,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        desc="Overall Progress",
        dynamic_ncols=True,
    )
    try:
        for idx, (rel_path, full_path, size, mtime) in enumerate(to_hash, 1):
            size_mb = size / (1024 ** 2)
            short_name = os.path.basename(rel_path)
            if len(short_name) > 42:
                short_name = short_name[:39] + "..."

            t0 = time.time()
            ed2k, crc32, md5, sha1 = compute_rhash_with_progress(
                str(full_path),
                size,
                desc=f"[{idx}/{len(to_hash)}] {short_name}",
                overall_bar=overall_bar,
            )
            mediainfo_obj = compute_mediainfo(str(full_path))
            dt = max(time.time() - t0, 0.001)
            speed = size_mb / dt

            existing_files[rel_path] = {
                "size": size,
                "mtime": round(mtime, 2),
                "ed2k": ed2k,
                "crc32": crc32,
                "md5": md5,
                "sha1": sha1,
                "mediainfo": mediainfo_obj,
            }
            save_index_and_meta_atomic(index_path, manifest)
            overall_bar.write(f"[{idx}/{len(to_hash)}] ✓ {rel_path} | ED2K: {ed2k} | {speed:.1f} MB/s ({dt:.1f} s)")
    finally:
        overall_bar.close()

    print(f"\n[+] Done! Index updated: {index_path} + {meta_path} (total files in index: {len(existing_files)})")
